# IOAI — 2026 Contest Ticket Extraction (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.listdir('data'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2026-contest-ticket-extraction/data.zip', 'd.zip')
    zipfile.ZipFile('d.zip').extractall('data')
print('데이터 준비:', sorted(os.listdir('data'))[:8])
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

In [ ]:
# === 모범답안2 전용: T4 에서 7B 를 돌리기 위한 4-bit 양자화 스택 ===
!pip install -q -U bitsandbytes accelerate
print("✅ bitsandbytes/accelerate 설치 완료 — 모델은 4-bit 로 로드됩니다(T4 적재 가능)")

# Ticket Extraction — 모범답안2 (Qwen2.5-7B LLM 베이스라인)

문제 개요가 말하는 **원 대회의 Qwen2.5-7B(LLM) 베이스라인**을 실제로 구현한 버전이다. 지도학습
모범답안(문자 n-gram TF-IDF + LinearSVC, 정확도 ≈0.877)과 달리, 여기서는 **라벨을 학습하지 않고**
지시형 LLM(**Qwen/Qwen2.5-7B-Instruct**)에 few-shot 프롬프트로 4필드를 **추출(zero/few-shot)** 한다.

**결과**: 전체 정확도 ≈ **0.83** (category 0.93 / product 1.0 / sentiment 0.78 / **priority 0.62**).
- **핵심 교훈**(문제 개요와 일치): 라벨 480개가 있으므로 *지도학습이 더 빠르고 살짝 더 정확*(0.877)하다.
  하지만 **잘 설계한 7B few-shot 은 학습 없이도 거의 근접**한다 — LLM 베이스라인의 실체를 보여준다.
- **프롬프트 설계가 전부**: 소박한 few-shot 만 쓰면 7B 도 0.78 에 그친다. ① 필드별 **판단 기준(rubric)**,
  ② **모든 라벨값을 덮는 stratified few-shot**(클래스당 2개), ③ product 는 제품목록으로 **정규화 매칭** →
  0.78 → **0.83**. priority 는 표현이 미묘해 zero-shot 로 가장 어렵다(라벨 규약 의존).

**주의(실행 환경)**: 7B 로드에 **GPU + bf16 ≈16GB** 필요(GB10/DGX 권장; T4 는 4-bit 양자화 필요). 모델은
`Qwen/Qwen2.5-7B-Instruct` 를 HuggingFace 에서 받는다. 모델 id 만 바꾸면 1.5B/3B(더 가벼움·정확도↓)로도 동작.


In [ ]:
import json, re, pandas as pd, torch
from transformers import AutoModelForCausalLM, AutoTokenizer

train = pd.read_csv("data/train.csv")     # id, text, category, priority, product, sentiment
test  = pd.read_csv("data/test.csv")      # id, text
FIELDS = ["category", "priority", "product", "sentiment"]
CATS  = ["billing", "technical", "account", "shipping", "refund"]
PRIOS = ["low", "medium", "high"]; SENTS = ["negative", "neutral", "positive"]
PRODUCTS = sorted(train["product"].astype(str).unique(), key=len, reverse=True)   # 제품 정규명 목록
prod_fb = train["product"].mode()[0]; cat_fb = train["category"].mode()[0]
prio_fb = train["priority"].mode()[0]; sent_fb = train["sentiment"].mode()[0]
print("train", len(train), "| test", len(test), "| 제품종류", len(PRODUCTS))


In [ ]:
# 프롬프트: 필드별 rubric + 클래스당 2개씩 stratified few-shot
shots = []
for f, vals in [("category", CATS), ("priority", PRIOS), ("sentiment", SENTS)]:
    for v in vals:
        shots += list(train[train[f] == v].sample(2, random_state=1).itertuples(index=False))
sh = pd.DataFrame(shots).drop_duplicates("id")
examples = "\n".join(
    f'문의: {r.text}\n출력: {{"category":"{r.category}","priority":"{r.priority}",'
    f'"product":"{r.product}","sentiment":"{r.sentiment}"}}' for r in sh.itertuples())

RUBRIC = (
 "판단 기준:\n"
 "- category: 결제/요금/카드=billing, 연결/작동/오류 등 기술문제=technical, "
 "로그인/계정/탈퇴/인증=account, 배송=shipping, 환불요청=refund.\n"
 "- priority: '급하지 않다/시간 될 때' 처럼 여유 표현이면 low, "
 "'세 번째 문의/긴급/빠른 답변/강한 불만' 이면 high, 그 외 긴급 표현이 없으면 medium.\n"
 "- sentiment: '좋아요/만족' 등 칭찬이면 positive, '짜증/화/실망' 등 불만 감정이면 negative, "
 "담백한 문의면 neutral.\n")
SYSTEM = (
 "당신은 고객 문의에서 정보를 추출하는 도우미입니다. 반드시 JSON 한 줄만 출력합니다.\n"
 f"category 는 {CATS} 중 하나. priority 는 {PRIOS} 중 하나. sentiment 는 {SENTS} 중 하나.\n"
 f"product 는 다음 제품목록 중 정확히 하나: {PRODUCTS}\n" + RUBRIC + "예시:\n" + examples)
print("few-shot 예시 수:", len(sh))


In [ ]:
# Qwen2.5-7B-Instruct 로드 (GPU/bf16) → 배치 생성
MODEL = "Qwen/Qwen2.5-7B-Instruct"          # 1.5B/3B 로 바꾸면 더 가벼움(정확도↓)
tok = AutoTokenizer.from_pretrained(MODEL)
from transformers import BitsAndBytesConfig
_bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                          bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=_bnb, device_map="auto").eval()  # T4: 4-bit

def generate(texts, bs=16):
    outs = []
    for i in range(0, len(texts), bs):
        chunk = texts[i:i+bs]
        prompts = [tok.apply_chat_template(
            [{"role": "system", "content": SYSTEM},
             {"role": "user", "content": f"문의: {t}\n출력(JSON만):"}],
            tokenize=False, add_generation_prompt=True) for t in chunk]
        enc = tok(prompts, return_tensors="pt", padding=True, padding_side="left").to("cuda")
        with torch.no_grad():
            g = model.generate(**enc, max_new_tokens=90, do_sample=False, pad_token_id=tok.eos_token_id)
        for j in range(len(chunk)):
            outs.append(tok.decode(g[j, enc.input_ids.shape[1]:], skip_special_tokens=True))
    return outs

raw = generate(list(test.text))
print("생성 완료:", len(raw))


In [ ]:
# JSON 파싱 + 허용값으로 정규화 (product 는 제품목록 매칭)
def parse(s):
    s = s.replace("```json", "").replace("```", "")
    m = re.search(r"\{.*?\}", s, re.S)
    try: return json.loads(m.group(0)) if m else {}
    except Exception: return {}

def norm(v, allowed, fb):
    v = str(v).strip().lower()
    for a in allowed:
        if a in v: return a
    return fb

def norm_prod(v, text):
    v = str(v).strip()
    for p in PRODUCTS:                     # LLM 출력이 정규명과 일치
        if p == v: return p
    for p in PRODUCTS:                     # 아니면 티켓 본문에 나타난 제품
        if p in str(text): return p
    for p in PRODUCTS:
        if p in v: return p
    return prod_fb

rows = []
for r, out in zip(test.itertuples(), raw):
    d = parse(out)
    vals = {"category": norm(d.get("category", ""), CATS, cat_fb),
            "priority": norm(d.get("priority", ""), PRIOS, prio_fb),
            "product":  norm_prod(d.get("product", ""), r.text),
            "sentiment": norm(d.get("sentiment", ""), SENTS, sent_fb)}
    for f in FIELDS:
        rows.append({"row_id": f"{r.id}__{f}", "value": vals[f]})
pd.DataFrame(rows).to_csv("submission.csv", index=False)
print("saved submission.csv", len(rows))


### 정리
- **Qwen2.5-7B few-shot 추출**로 학습 없이 전체 정확도 ≈ **0.83**. 지도학습 모범답안(0.877)에 근접.
- **프롬프트가 성능을 좌우**: rubric + 전 클래스 stratified few-shot + product 정규화 매칭이 0.78→0.83 을 만든다.
- **필드별 난이도**: product(정규목록 매칭) 1.0, category 0.93, sentiment 0.78, **priority 0.62(가장 어려움 —
  긴급도 표현이 미묘하고 라벨 규약 의존)**. 라벨이 충분하면 지도학습이, 라벨이 없거나 새 필드가 추가되면
  LLM 추출이 유리하다(라벨링 불필요·즉시 확장).
- **더 끌어올리려면**: 앙상블(여러 seed 투표), self-consistency, priority/sentiment 만 소량 지도보정,
  또는 더 큰 모델(72B)·소량 파인튜닝.


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)